In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
import sys
from pathlib import Path

# Make the notebook runnable from either the repository root or this folder.
project_root = next(
    path for path in [Path.cwd(), *Path.cwd().parents]
    if (path / "src" / "tinytorch").is_dir()
)
if str(project_root / "src") not in sys.path:
    sys.path.insert(0, str(project_root / "src"))

# Data loader experiments

## tl;dr

All correctness assertions pass. TinyTorch and PyTorch produced identical sequential batches with zero feature or label error. On the 4,096-sample CPU fixture, TinyTorch took 10.288 ms/epoch versus PyTorch's 20.606 ms/epoch (0.50x, or 398,122 TinyTorch samples/s). These timings apply only to this small, single-process in-memory workload.

## Context & Methods

All correctness checks use deterministic synthetic inputs and fixed random seeds. Efficiency timings use pre-created in-memory tensors, warm-up epochs, and the median of repeated full-epoch runs; they measure Python iteration plus collation, not disk I/O, multiprocessing, or model computation.

### Key assumptions

- Both loaders use one process, no shuffling, the same `float32` data, and the same batch size.
- PyTorch and TinyTorch expose different tensor wrappers, so equality is checked on their underlying NumPy values.
- Timings describe this machine and workload only; they are not a general performance guarantee.

In [3]:
import random
from statistics import median
from time import perf_counter

import numpy as np
import torch
from torch.utils.data import DataLoader as TorchDataLoader
from torch.utils.data import TensorDataset as TorchTensorDataset

from tinytorch.data_loader import (
    Compose,
    DataLoader,
    RandomCrop,
    RandomHorizontalFlip,
    TensorDataset,
    _pad_image,
)
from tinytorch.tensor import Tensor_CP


def assert_raises(exception_type, operation):
    """Assert that operation raises exactly the expected exception family."""
    try:
        operation()
    except exception_type as error:
        return error
    except Exception as error:
        raise AssertionError(
            f"Expected {exception_type.__name__}, got {type(error).__name__}"
        ) from error
    raise AssertionError(f"Expected {exception_type.__name__}, but nothing was raised")


loader_checks = {}

## TensorDataset

A tensor dataset stores aligned tensors and returns one tuple of tensors per sample. These small checks establish the input contract used by the loader experiments below.

In [4]:
features = Tensor_CP([[1, 2], [3, 4], [5, 6], [7, 8]])
labels = Tensor_CP([0, 1, 1, 0])
dataset = TensorDataset(features, labels)

assert len(dataset) == 4  # samples, not number of stored tensors
first_sample = dataset[0]
assert isinstance(first_sample, tuple)
assert len(first_sample) == 2
assert all(isinstance(value, Tensor_CP) for value in first_sample)
assert np.array_equal(first_sample[0].data, np.array([1, 2], dtype=np.float32))
assert first_sample[1].data.item() == 0
print("PASS: four aligned samples return feature-label Tensor_CP pairs")

PASS: four aligned samples return feature-label Tensor_CP pairs


### Invalid TensorDataset inputs

Stored tensors must have matching first dimensions. Plain lists, NumPy arrays, and scalar tensors are rejected because they do not satisfy the dataset contract.

In [5]:
assert_raises(
    ValueError,
    lambda: TensorDataset(Tensor_CP([[1], [2], [3], [4]]), Tensor_CP([0, 1, 2])),
)
assert_raises(TypeError, lambda: TensorDataset([1, 2, 3, 4]))
assert_raises(TypeError, lambda: TensorDataset(np.array([1, 2, 3, 4])))
assert_raises(ValueError, lambda: TensorDataset(Tensor_CP(42)))
print("PASS: mismatched lengths, non-tensors, and scalar tensors are rejected")

PASS: mismatched lengths, non-tensors, and scalar tensors are rejected


### Empty and single-tensor datasets

In [6]:
empty_features = Tensor_CP(np.empty((0, 3), dtype=np.float32))
empty_labels = Tensor_CP(np.empty((0,), dtype=np.float32))
empty_dataset = TensorDataset(empty_features, empty_labels)
assert len(empty_dataset) == 0

single_dataset = TensorDataset(Tensor_CP([10, 20, 30, 40]))
assert len(single_dataset) == 4
single_sample = single_dataset[0]
assert isinstance(single_sample, tuple) and len(single_sample) == 1
assert isinstance(single_sample[0], Tensor_CP)
assert single_sample[0].data.item() == 10
print("PASS: aligned empty tensors and a single tensor both work")

PASS: aligned empty tensors and a single tensor both work


## DataLoader

The main fixture has 11 samples. Each feature begins with a unique sample ID, and its label follows `label = 3 * sample_id + 5`. That relationship lets us detect any feature-label misalignment after shuffling.

In [7]:
sample_ids = np.arange(11, dtype=np.float32)
feature_values = np.column_stack((sample_ids, sample_ids * 10 + 1))
label_values = sample_ids * 3 + 5
batch_dataset = TensorDataset(Tensor_CP(feature_values), Tensor_CP(label_values))

print(f"Samples:       {len(batch_dataset)}")
print(f"Feature shape: {feature_values.shape}")
print(f"Label shape:   {label_values.shape}")

Samples:       11
Feature shape: (11, 2)
Label shape:   (11,)


### Batch count, sizes, and shapes

With 11 samples and `batch_size=4`, ceiling division gives three batches. The final incomplete batch must be retained, so the first dimensions are `4`, `4`, and `3`. Collation adds that batch dimension without changing each sample's remaining dimensions.

In [8]:
sequential_loader = DataLoader(batch_dataset, batch_size=4, shuffle=False)
sequential_batches = list(sequential_loader)

batch_sizes = [batch_features.shape[0] for batch_features, _ in sequential_batches]
feature_shapes = [batch_features.shape for batch_features, _ in sequential_batches]
label_shapes = [batch_labels.shape for _, batch_labels in sequential_batches]

assert batch_sizes == [4, 4, 3]
assert len(sequential_loader) == 3
assert feature_shapes == [(4, 2), (4, 2), (3, 2)]
assert label_shapes == [(4,), (4,), (3,)]

loader_checks["11 samples with batch size 4 produces batch sizes 4, 4, 3"] = True
loader_checks["len(loader) returns 3"] = True
loader_checks["Batch feature and label shapes are correct"] = True

print(f"Batch sizes:    {batch_sizes}")
print(f"Feature shapes: {feature_shapes}")
print(f"Label shapes:   {label_shapes}")
print(f"len(loader):    {len(sequential_loader)}")
print("PASS: full and partial batches have the expected sizes and shapes")

Batch sizes:    [4, 4, 3]
Feature shapes: [(4, 2), (4, 2), (3, 2)]
Label shapes:   [(4,), (4,), (3,)]
len(loader):    3
PASS: full and partial batches have the expected sizes and shapes


### Sequential order with `shuffle=False`

When shuffling is disabled, concatenating the sample IDs from all batches must reproduce the dataset's original order exactly.

In [9]:
sequential_order = np.concatenate(
    [batch_features.data[:, 0] for batch_features, _ in sequential_batches]
).astype(int)

assert np.array_equal(sequential_order, np.arange(11))
loader_checks["shuffle=False preserves sequential order"] = True

print(f"Observed sample IDs: {sequential_order.tolist()}")
print("PASS: shuffle=False preserves sequential order")

Observed sample IDs: [0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10]
PASS: shuffle=False preserves sequential order


### Shuffling keeps complete samples aligned

The loader shuffles dataset indices, not tensors independently. We run two epochs and verify both the feature-label relationship and exact once-per-epoch coverage. A fixed seed makes this notebook's displayed orders reproducible.

In [10]:
random.seed(7)
shuffled_loader = DataLoader(batch_dataset, batch_size=4, shuffle=True)
epoch_orders = []

for epoch in range(2):
    order = []
    for batch_features, batch_labels in shuffled_loader:
        ids = batch_features.data[:, 0]
        expected_labels = ids * 3 + 5
        assert np.array_equal(batch_labels.data, expected_labels)
        order.extend(ids.astype(int).tolist())

    unique_ids, counts = np.unique(order, return_counts=True)
    assert np.array_equal(unique_ids, np.arange(11))
    assert np.array_equal(counts, np.ones(11, dtype=int))
    epoch_orders.append(order)

assert epoch_orders[0] != list(range(11))
assert epoch_orders[0] != epoch_orders[1]
loader_checks["shuffle=True preserves feature-label alignment"] = True
loader_checks["Every sample appears once per epoch"] = True

for epoch, order in enumerate(epoch_orders, start=1):
    print(f"Epoch {epoch}: {order}")
print("PASS: shuffling preserves pairs and visits every sample exactly once per epoch")

Epoch 1: [1, 10, 3, 9, 8, 4, 7, 0, 6, 2, 5]
Epoch 2: [2, 5, 4, 7, 10, 6, 9, 1, 0, 3, 8]
PASS: shuffling preserves pairs and visits every sample exactly once per epoch


### Empty dataset

An empty dataset has no valid batch starts: its loader length is zero and iteration yields nothing.

In [11]:
empty_loader = DataLoader(empty_dataset, batch_size=4, shuffle=False)
empty_batches = list(empty_loader)

assert len(empty_loader) == 0
assert empty_batches == []
loader_checks["Empty dataset produces zero batches"] = True

print(f"len(empty_loader): {len(empty_loader)}")
print(f"Batches produced:   {len(empty_batches)}")
print("PASS: an empty dataset produces zero batches")

len(empty_loader): 0
Batches produced:   0
PASS: an empty dataset produces zero batches


### `batch_size` validation

A batch size must be a positive built-in integer. Zero and negative integers describe no usable group size, while floats, strings, `None`, and Booleans are the wrong type. Booleans need an explicit type check because Python otherwise treats `bool` as a subclass of `int`.

In [12]:
nonpositive_batch_sizes = [0, -1, -4]
for invalid_size in nonpositive_batch_sizes:
    assert_raises(
        ValueError,
        lambda invalid_size=invalid_size: DataLoader(batch_dataset, invalid_size),
    )

wrong_type_batch_sizes = [4.0, "4", None, True, False]
for invalid_size in wrong_type_batch_sizes:
    assert_raises(
        TypeError,
        lambda invalid_size=invalid_size: DataLoader(batch_dataset, invalid_size),
    )

loader_checks["batch_size=0 and negative values raise ValueError"] = True
loader_checks["Float, string, None, True, and False batch sizes raise TypeError"] = True

print(f"ValueError inputs: {[repr(value) for value in nonpositive_batch_sizes]}")
print(f"TypeError inputs:  {[repr(value) for value in wrong_type_batch_sizes]}")
print("PASS: batch_size accepts only positive built-in integers")

ValueError inputs: ['0', '-1', '-4']


TypeError inputs:  ['4.0', "'4'", 'None', 'True', 'False']
PASS: batch_size accepts only positive built-in integers


### `shuffle` validation

`shuffle` is a switch rather than a truthiness test, so only the built-in Boolean values `True` and `False` are accepted.

In [13]:
non_boolean_shuffle_values = [0, 1, 0.0, "yes", None, [], np.bool_(True)]
for invalid_shuffle in non_boolean_shuffle_values:
    assert_raises(
        TypeError,
        lambda invalid_shuffle=invalid_shuffle: DataLoader(
            batch_dataset, batch_size=4, shuffle=invalid_shuffle
        ),
    )

loader_checks["Non-Boolean shuffle values raise TypeError"] = True

print(f"Rejected values: {[repr(value) for value in non_boolean_shuffle_values]}")
print("PASS: shuffle accepts only True or False")

Rejected values: ['0', '1', '0.0', "'yes'", 'None', '[]', 'np.True_']
PASS: shuffle accepts only True or False


## DataLoader results

This cell checks off the requested behaviors from the values recorded by the executed assertion cells above.

In [14]:
expected_loader_checks = [
    "11 samples with batch size 4 produces batch sizes 4, 4, 3",
    "len(loader) returns 3",
    "Batch feature and label shapes are correct",
    "shuffle=False preserves sequential order",
    "shuffle=True preserves feature-label alignment",
    "Every sample appears once per epoch",
    "Empty dataset produces zero batches",
    "batch_size=0 and negative values raise ValueError",
    "Float, string, None, True, and False batch sizes raise TypeError",
    "Non-Boolean shuffle values raise TypeError",
]

assert list(loader_checks) == expected_loader_checks
assert all(loader_checks.values())

for check in expected_loader_checks:
    print(f"[x] {check}")
print(f"\nAll {len(expected_loader_checks)} requested DataLoader checks passed.")

[x] 11 samples with batch size 4 produces batch sizes 4, 4, 3
[x] len(loader) returns 3
[x] Batch feature and label shapes are correct
[x] shuffle=False preserves sequential order
[x] shuffle=True preserves feature-label alignment
[x] Every sample appears once per epoch
[x] Empty dataset produces zero batches
[x] batch_size=0 and negative values raise ValueError
[x] Float, string, None, True, and False batch sizes raise TypeError
[x] Non-Boolean shuffle values raise TypeError

All 10 requested DataLoader checks passed.


## RandomHorizontalFlip

`RandomHorizontalFlip` accepts NumPy arrays and `Tensor_CP` images. It preserves the input container type and flips only the horizontal spatial axis. The current format convention chooses that axis as follows:

| Image layout | Detection rule | Horizontal axis |
|---|---|---:|
| 2D `(height, width)` | `ndim == 2` | `-1` |
| Channels-first `(channels, height, width)` | `ndim == 3` and first dimension `<= 4` | `-1` |
| Channels-last `(height, width, channels)` | `ndim == 3` and first dimension `> 4` | `-2` |

### Always flip a 2D NumPy image

With `p=1`, every call must reverse the width axis. A snapshot verifies that producing the transformed array does not mutate the input.

In [15]:
flip_checks = {}
image_2d = np.array([[1, 2, 3], [4, 5, 6]])
image_2d_before = image_2d.copy()
expected_2d_flip = np.array([[3, 2, 1], [6, 5, 4]])

always_flip = RandomHorizontalFlip(p=1)
flipped_2d = always_flip(image_2d)

assert isinstance(flipped_2d, np.ndarray)
assert flipped_2d.shape == (2, 3)
assert np.array_equal(flipped_2d, expected_2d_flip)
assert np.array_equal(image_2d, image_2d_before)
assert not np.shares_memory(flipped_2d, image_2d)
flip_checks["Always flip a 2D NumPy image without modifying the input"] = True

print("Input:")
print(image_2d)
print("Flipped:")
print(flipped_2d)
print("PASS: output is a shape-preserving NumPy array and the input is unchanged")

Input:


[[1 2 3]
 [4 5 6]]
Flipped:
[[3 2 1]
 [6 5 4]]
PASS: output is a shape-preserving NumPy array and the input is unchanged


### Never flip

With `p=0`, the random draw can never trigger the transform, so the returned values must equal the original image.

In [16]:
never_flip = RandomHorizontalFlip(p=0)
unchanged_2d = never_flip(image_2d)

assert isinstance(unchanged_2d, np.ndarray)
assert np.array_equal(unchanged_2d, image_2d)
flip_checks["Never flip when p=0"] = True

print(unchanged_2d)
print("PASS: p=0 preserves the original image values")

[[1 2 3]
 [4 5 6]]
PASS: p=0 preserves the original image values


### Preserve `Tensor_CP`

A tensor input must produce a new `Tensor_CP`, while its source data remains untouched.

In [17]:
tensor_image = Tensor_CP(image_2d)
tensor_before = tensor_image.data.copy()
flipped_tensor = always_flip(tensor_image)

assert isinstance(flipped_tensor, Tensor_CP)
assert flipped_tensor.shape == (2, 3)
assert np.array_equal(flipped_tensor.data, expected_2d_flip)
assert np.array_equal(tensor_image.data, tensor_before)
assert flipped_tensor is not tensor_image
flip_checks["Preserve Tensor_CP type, shape, and source data"] = True

print(f"Output type:  {type(flipped_tensor).__name__}")
print(f"Output shape: {flipped_tensor.shape}")
print(flipped_tensor.data)
print("PASS: Tensor_CP is preserved and the original tensor is unchanged")

Output type:  Tensor_CP
Output shape: (2, 3)
[[3. 2. 1.]
 [6. 5. 4.]]
PASS: Tensor_CP is preserved and the original tensor is unchanged


### Channels-first image

For `(channels, height, width) = (3, 5, 6)`, the last axis is width, so the result should equal `np.flip(image, axis=-1)`.

In [18]:
channels_first = np.arange(3 * 5 * 6).reshape(3, 5, 6)
flipped_channels_first = always_flip(channels_first)

assert flipped_channels_first.shape == (3, 5, 6)
assert np.array_equal(
    flipped_channels_first, np.flip(channels_first, axis=-1)
)
flip_checks["Flip channels-first image along axis -1"] = True

print(f"Input shape:  {channels_first.shape}")
print(f"Output shape: {flipped_channels_first.shape}")
print("PASS: channels-first image flips along width axis -1")

Input shape:  (3, 5, 6)


Output shape: (3, 5, 6)
PASS: channels-first image flips along width axis -1


### Channels-last image

For `(height, width, channels) = (5, 6, 3)`, the second-to-last axis is width. Height is deliberately greater than four so the format convention detects channels-last layout.

In [19]:
channels_last = np.arange(5 * 6 * 3).reshape(5, 6, 3)
flipped_channels_last = always_flip(channels_last)

assert channels_last.shape[0] > 4
assert flipped_channels_last.shape == (5, 6, 3)
assert np.array_equal(
    flipped_channels_last, np.flip(channels_last, axis=-2)
)
flip_checks["Flip channels-last image along axis -2"] = True

print(f"Input shape:  {channels_last.shape}")
print(f"Output shape: {flipped_channels_last.shape}")
print("PASS: channels-last image flips along width axis -2")

Input shape:  (5, 6, 3)


Output shape: (5, 6, 3)
PASS: channels-last image flips along width axis -2


### Probability validation

The default probability is `0.5`. Numeric probabilities outside `[0, 1]` are invalid values; Booleans and non-numeric values are invalid types. Booleans are tested explicitly because Python treats `bool` as a subclass of `int`.

In [20]:
default_flip = RandomHorizontalFlip()
assert default_flip.p == 0.5

out_of_range_probabilities = [-0.1, 1.1]
for invalid_probability in out_of_range_probabilities:
    assert_raises(
        ValueError,
        lambda invalid_probability=invalid_probability: RandomHorizontalFlip(
            invalid_probability
        ),
    )

wrong_type_probabilities = [True, False, "0.5", None]
for invalid_probability in wrong_type_probabilities:
    assert_raises(
        TypeError,
        lambda invalid_probability=invalid_probability: RandomHorizontalFlip(
            invalid_probability
        ),
    )

flip_checks["Validate probability and keep the default at 0.5"] = True

print(f"Default p:         {default_flip.p}")
print(f"ValueError inputs: {[repr(value) for value in out_of_range_probabilities]}")
print(f"TypeError inputs:  {[repr(value) for value in wrong_type_probabilities]}")
print("PASS: probability defaults and validation are correct")

Default p:         0.5


ValueError inputs: ['-0.1', '1.1']
TypeError inputs:  ['True', 'False', "'0.5'", 'None']
PASS: probability defaults and validation are correct


### Input validation

Only NumPy arrays and `Tensor_CP` objects are accepted. Image data must be two- or three-dimensional; one-dimensional arrays, four-dimensional batches, and scalar tensors do not describe one supported image.

In [21]:
wrong_type_images = [[1, 2, 3], (1, 2, 3), "image", 42]
for invalid_image in wrong_type_images:
    assert_raises(
        TypeError,
        lambda invalid_image=invalid_image: always_flip(invalid_image),
    )

wrong_dimension_images = [
    np.arange(3),
    np.zeros((1, 2, 3, 4)),
    Tensor_CP(42),
]
for invalid_image in wrong_dimension_images:
    assert_raises(
        ValueError,
        lambda invalid_image=invalid_image: always_flip(invalid_image),
    )

flip_checks["Reject unsupported input types and dimensions"] = True

print("TypeError inputs: Python list, tuple, string, and scalar number")
print("ValueError inputs: 1D array, 4D array, and scalar Tensor_CP")
print("PASS: unsupported image inputs are rejected")

TypeError inputs: Python list, tuple, string, and scalar number
ValueError inputs: 1D array, 4D array, and scalar Tensor_CP
PASS: unsupported image inputs are rejected


### Randomness

With `p=0.5`, repeated calls should produce both outcomes. The asymmetric image makes flipped and unchanged results unambiguous, while a fixed Python seed makes the displayed counts reproducible.

In [22]:
random.seed(7)
random_flip = RandomHorizontalFlip(p=0.5)
trial_count = 100
flipped_count = 0
unchanged_count = 0

for _ in range(trial_count):
    result = random_flip(image_2d)
    if np.array_equal(result, expected_2d_flip):
        flipped_count += 1
    elif np.array_equal(result, image_2d):
        unchanged_count += 1
    else:
        raise AssertionError("Transform returned an unexpected image")

assert flipped_count + unchanged_count == trial_count
assert 25 <= flipped_count <= 75
assert 25 <= unchanged_count <= 75
flip_checks["Observe both outcomes across 100 seeded calls with p=0.5"] = True

print(f"Trials:    {trial_count}")
print(f"Flipped:   {flipped_count}")
print(f"Unchanged: {unchanged_count}")
print("PASS: p=0.5 produces both outcomes across repeated calls")

Trials:    100
Flipped:   54
Unchanged: 46
PASS: p=0.5 produces both outcomes across repeated calls


## RandomHorizontalFlip results

This checklist is generated only after every transform assertion above has passed.

In [23]:
expected_flip_checks = [
    "Always flip a 2D NumPy image without modifying the input",
    "Never flip when p=0",
    "Preserve Tensor_CP type, shape, and source data",
    "Flip channels-first image along axis -1",
    "Flip channels-last image along axis -2",
    "Validate probability and keep the default at 0.5",
    "Reject unsupported input types and dimensions",
    "Observe both outcomes across 100 seeded calls with p=0.5",
]

assert list(flip_checks) == expected_flip_checks
assert all(flip_checks.values())

for check in expected_flip_checks:
    print(f"[x] {check}")
print(f"\nAll {len(expected_flip_checks)} requested RandomHorizontalFlip checks passed.")

[x] Always flip a 2D NumPy image without modifying the input
[x] Never flip when p=0
[x] Preserve Tensor_CP type, shape, and source data
[x] Flip channels-first image along axis -1
[x] Flip channels-last image along axis -2
[x] Validate probability and keep the default at 0.5
[x] Reject unsupported input types and dimensions
[x] Observe both outcomes across 100 seeded calls with p=0.5

All 8 requested RandomHorizontalFlip checks passed.


## Image padding

`_pad_image` adds a zero border to spatial dimensions while leaving any channel dimension untouched. It follows the same layout convention as the horizontal flip:

| Input layout | Spatial axes | Input shape | With `padding=2` |
|---|---|---:|---:|
| 2D | `(0, 1)` | `(8, 8)` | `(12, 12)` |
| Channels-first | `(1, 2)` | `(3, 8, 8)` | `(3, 12, 12)` |
| Channels-last | `(0, 1)` | `(8, 8, 3)` | `(12, 12, 3)` |

In [24]:
padding_checks = {}


def verify_zero_padding(image, padding, spatial_axes):
    """Return the padded image after checking shape, border, centre, and input safety."""
    image_before = image.copy()
    padded = _pad_image(image, padding)

    expected_shape = list(image.shape)
    for axis in spatial_axes:
        expected_shape[axis] += 2 * padding

    centre_slices = [slice(None)] * image.ndim
    spatial_slice = slice(padding, -padding) if padding else slice(None)
    for axis in spatial_axes:
        centre_slices[axis] = spatial_slice
    centre_slices = tuple(centre_slices)

    border_mask = np.ones(padded.shape, dtype=bool)
    border_mask[centre_slices] = False

    assert isinstance(padded, np.ndarray)
    assert padded.shape == tuple(expected_shape)
    assert np.array_equal(padded[centre_slices], image)
    assert np.all(padded[border_mask] == 0)
    assert np.array_equal(image, image_before)
    return padded


padding = 2
padding_2d = np.arange(1, 8 * 8 + 1).reshape(8, 8)
padding_channels_first = np.arange(1, 3 * 8 * 8 + 1).reshape(3, 8, 8)
padding_channels_last = np.arange(1, 8 * 8 * 3 + 1).reshape(8, 8, 3)

padded_2d = verify_zero_padding(padding_2d, padding, spatial_axes=(0, 1))
padded_channels_first = verify_zero_padding(
    padding_channels_first, padding, spatial_axes=(1, 2)
)
padded_channels_last = verify_zero_padding(
    padding_channels_last, padding, spatial_axes=(0, 1)
)

assert padded_channels_first.shape[0] == padding_channels_first.shape[0]
assert padded_channels_last.shape[-1] == padding_channels_last.shape[-1]

padding_checks["(8, 8) with padding 2 becomes (12, 12)"] = True
padding_checks["(3, 8, 8) becomes (3, 12, 12)"] = True
padding_checks["(8, 8, 3) becomes (12, 12, 3)"] = True
padding_checks["Padding borders are zero"] = True
padding_checks["The centre contains the original data"] = True
padding_checks["Channel dimensions remain unchanged"] = True
padding_checks["Original arrays are not modified"] = True

print(f"{'layout':<18} | {'input':<10} | {'output':<12}")
print("-" * 46)
for layout, source, result in [
    ("2D", padding_2d, padded_2d),
    ("channels-first", padding_channels_first, padded_channels_first),
    ("channels-last", padding_channels_last, padded_channels_last),
]:
    print(f"{layout:<18} | {str(source.shape):<10} | {str(result.shape):<12}")
print("PASS: shapes, zero borders, centred data, channels, and inputs are correct")

layout             | input      | output      
----------------------------------------------
2D                 | (8, 8)     | (12, 12)    
channels-first     | (3, 8, 8)  | (3, 12, 12) 
channels-last      | (8, 8, 3)  | (12, 12, 3) 
PASS: shapes, zero borders, centred data, channels, and inputs are correct


### Zero padding

A padding width of zero is valid and must preserve both shape and values.

In [25]:
zero_padded = _pad_image(padding_2d, padding=0)

assert zero_padded.shape == padding_2d.shape
assert np.array_equal(zero_padded, padding_2d)
padding_checks["Padding 0 preserves shape and values"] = True

print(f"Input shape:  {padding_2d.shape}")
print(f"Output shape: {zero_padded.shape}")
print("PASS: padding=0 preserves shape and values")

Input shape:  (8, 8)
Output shape: (8, 8)
PASS: padding=0 preserves shape and values


### Padding parameter validation

Padding must be a non-negative built-in integer. Negative integers are invalid values; Booleans and all other non-integer values are invalid types.

In [26]:
negative_padding_values = [-1, -3]
for invalid_padding in negative_padding_values:
    assert_raises(
        ValueError,
        lambda invalid_padding=invalid_padding: _pad_image(
            padding_2d, invalid_padding
        ),
    )

wrong_type_padding_values = [True, False, 2.0, "2", None]
for invalid_padding in wrong_type_padding_values:
    assert_raises(
        TypeError,
        lambda invalid_padding=invalid_padding: _pad_image(
            padding_2d, invalid_padding
        ),
    )

padding_checks["Negative padding raises ValueError"] = True
padding_checks["Boolean and non-integer padding raise TypeError"] = True

print(f"ValueError inputs: {[repr(value) for value in negative_padding_values]}")
print(f"TypeError inputs:  {[repr(value) for value in wrong_type_padding_values]}")
print("PASS: invalid padding values and types are rejected")

ValueError inputs: ['-1', '-3']
TypeError inputs:  ['True', 'False', '2.0', "'2'", 'None']
PASS: invalid padding values and types are rejected


### Dimension validation

The helper pads one image at a time, so only two- and three-dimensional arrays are supported.

In [27]:
wrong_dimension_padding_inputs = [
    np.arange(8),
    np.zeros((1, 2, 3, 4)),
]
for invalid_data in wrong_dimension_padding_inputs:
    assert_raises(
        ValueError,
        lambda invalid_data=invalid_data: _pad_image(invalid_data, padding=2),
    )

padding_checks["1D and 4D data raise ValueError"] = True

print(f"Rejected shapes: {[data.shape for data in wrong_dimension_padding_inputs]}")
print("PASS: 1D and 4D arrays are rejected")

Rejected shapes: [(8,), (1, 2, 3, 4)]
PASS: 1D and 4D arrays are rejected


## Image padding results

This checklist is generated only after every padding assertion above has passed.

In [28]:
expected_padding_checks = [
    "(8, 8) with padding 2 becomes (12, 12)",
    "(3, 8, 8) becomes (3, 12, 12)",
    "(8, 8, 3) becomes (12, 12, 3)",
    "Padding borders are zero",
    "The centre contains the original data",
    "Channel dimensions remain unchanged",
    "Original arrays are not modified",
    "Padding 0 preserves shape and values",
    "Negative padding raises ValueError",
    "Boolean and non-integer padding raise TypeError",
    "1D and 4D data raise ValueError",
]

assert list(padding_checks) == expected_padding_checks
assert all(padding_checks.values())

for check in expected_padding_checks:
    print(f"[x] {check}")
print(f"\nAll {len(expected_padding_checks)} requested image-padding checks passed.")

[x] (8, 8) with padding 2 becomes (12, 12)
[x] (3, 8, 8) becomes (3, 12, 12)
[x] (8, 8, 3) becomes (12, 12, 3)
[x] Padding borders are zero
[x] The centre contains the original data
[x] Channel dimensions remain unchanged
[x] Original arrays are not modified
[x] Padding 0 preserves shape and values
[x] Negative padding raises ValueError
[x] Boolean and non-integer padding raise TypeError
[x] 1D and 4D data raise ValueError

All 11 requested image-padding checks passed.


## RandomCrop

`RandomCrop` optionally pads an image, samples a valid top-left coordinate, and returns a crop in the same HW, CHW, or HWC layout. These experiments reuse the numbered padding fixtures above so every crop is asymmetric and input mutation is easy to detect.

### Identity case

With `padding=0` and a target equal to the image size, the only valid crop starts at `(0, 0)` and must contain identical data.

In [29]:
crop_checks = {}
identity_source = padding_2d.copy()
identity_crop = RandomCrop(size=identity_source.shape, padding=0)
identity_output = identity_crop(identity_source)

assert identity_output.shape == identity_source.shape
assert np.array_equal(identity_output, identity_source)
crop_checks["padding=0 with target equal to image size returns identical data"] = True

print(f"Input shape:  {identity_source.shape}")
print(f"Output shape: {identity_output.shape}")
print("PASS: the full-size zero-padding crop is identical to its input")

Input shape:  (8, 8)
Output shape: (8, 8)
PASS: the full-size zero-padding crop is identical to its input


### Layouts, container types, and input safety

A rectangular `(5, 6)` crop changes only the spatial dimensions. NumPy inputs remain NumPy arrays, a `Tensor_CP` input remains a tensor, and none of the source images may be modified.

In [30]:
rectangular_crop = RandomCrop(size=(5, 6), padding=0)
crop_sources = {
    "2D": padding_2d,
    "CHW": padding_channels_first,
    "HWC": padding_channels_last,
}
crop_sources_before = {name: image.copy() for name, image in crop_sources.items()}

random.seed(11)
crop_outputs = {name: rectangular_crop(image) for name, image in crop_sources.items()}

assert crop_outputs["2D"].shape == (5, 6)
assert crop_outputs["CHW"].shape == (3, 5, 6)
assert crop_outputs["HWC"].shape == (5, 6, 3)
assert all(isinstance(output, np.ndarray) for output in crop_outputs.values())
assert all(
    np.array_equal(image, crop_sources_before[name])
    for name, image in crop_sources.items()
)

tensor_crop_source = Tensor_CP(padding_2d)
tensor_crop_before = tensor_crop_source.data.copy()
random.seed(11)
tensor_crop_output = rectangular_crop(tensor_crop_source)

assert isinstance(tensor_crop_output, Tensor_CP)
assert tensor_crop_output.shape == (5, 6)
assert np.array_equal(tensor_crop_source.data, tensor_crop_before)

crop_checks.update({
    "2D (H, W) produces (target_h, target_w)": True,
    "CHW produces (C, target_h, target_w)": True,
    "HWC produces (target_h, target_w, C)": True,
    "NumPy input returns NumPy": True,
    "Tensor_CP input returns Tensor_CP": True,
    "Original input remains unchanged": True,
})

print(f"{'layout':<8} | {'input':<10} | {'output':<10} | {'type'}")
print("-" * 49)
for name in crop_sources:
    print(
        f"{name:<8} | {str(crop_sources[name].shape):<10} | "
        f"{str(crop_outputs[name].shape):<10} | {type(crop_outputs[name]).__name__}"
    )
print(f"Tensor_CP output shape: {tensor_crop_output.shape}")
print("PASS: layouts, return types, and original inputs are correct")

layout   | input      | output     | type
-------------------------------------------------
2D       | (8, 8)     | (5, 6)     | ndarray
CHW      | (3, 8, 8)  | (3, 5, 6)  | ndarray
HWC      | (8, 8, 3)  | (5, 6, 3)  | ndarray
Tensor_CP output shape: (5, 6)
PASS: layouts, return types, and original inputs are correct


### Integer and tuple sizes

An integer is normalized to a square `(size, size)` target, while a two-integer tuple keeps independent height and width values.

In [31]:
square_crop = RandomCrop(size=5, padding=0)
random.seed(19)
square_output = square_crop(padding_2d)

assert square_crop.size == (5, 5)
assert square_output.shape == (5, 5)
assert rectangular_crop.size == (5, 6)
assert crop_outputs["2D"].shape == (5, 6)

crop_checks["Integer size becomes a square crop"] = True
crop_checks["Tuple size produces a rectangular crop"] = True

print(f"Integer size 5 -> stored size {square_crop.size}, output {square_output.shape}")
print(f"Tuple size (5, 6) -> stored size {rectangular_crop.size}, output {crop_outputs['2D'].shape}")
print("PASS: integer and tuple size forms produce the expected crop shapes")

Integer size 5 -> stored size (5, 5), output (5, 5)
Tuple size (5, 6) -> stored size (5, 6), output (5, 6)
PASS: integer and tuple size forms produce the expected crop shapes


### Seed reproducibility

`RandomCrop` uses Python's `random` module. Resetting the same seed before each call must reproduce the same crop from an asymmetric image.

In [32]:
random.seed(37)
seeded_crop_a = rectangular_crop(padding_2d)
random.seed(37)
seeded_crop_b = rectangular_crop(padding_2d)

assert np.array_equal(seeded_crop_a, seeded_crop_b)
crop_checks["Resetting the same random seed produces the same crop"] = True

print(seeded_crop_a)
print("PASS: resetting seed 37 reproduces the same crop")

[[ 3  4  5  6  7  8]
 [11 12 13 14 15 16]
 [19 20 21 22 23 24]
 [27 28 29 30 31 32]
 [35 36 37 38 39 40]]
PASS: resetting seed 37 reproduces the same crop


### Oversized targets and constructor validation

The target must fit inside the padded spatial dimensions. Constructor checks also distinguish invalid values from invalid types for both `size` and `padding`.

In [33]:
oversized_crop = RandomCrop(size=(11, 8), padding=1)
assert_raises(ValueError, lambda: oversized_crop(padding_2d))
crop_checks["Target larger than the padded image raises ValueError"] = True

invalid_crop_arguments = [
    (ValueError, {"size": 0}),
    (ValueError, {"size": -1}),
    (ValueError, {"size": (3,)}),
    (ValueError, {"size": (3, 4, 5)}),
    (ValueError, {"size": (3, 0)}),
    (ValueError, {"size": (-1, 3)}),
    (TypeError, {"size": True}),
    (TypeError, {"size": False}),
    (TypeError, {"size": 3.5}),
    (TypeError, {"size": "3"}),
    (TypeError, {"size": None}),
    (TypeError, {"size": [3, 3]}),
    (TypeError, {"size": (3, 2.5)}),
    (ValueError, {"size": 3, "padding": -1}),
    (TypeError, {"size": 3, "padding": True}),
    (TypeError, {"size": 3, "padding": False}),
    (TypeError, {"size": 3, "padding": 1.5}),
    (TypeError, {"size": 3, "padding": "1"}),
    (TypeError, {"size": 3, "padding": None}),
]

for exception_type, arguments in invalid_crop_arguments:
    assert_raises(
        exception_type,
        lambda arguments=arguments: RandomCrop(**arguments),
    )

crop_checks["Invalid sizes and padding values raise the intended errors"] = True

value_error_count = sum(error is ValueError for error, _ in invalid_crop_arguments)
type_error_count = sum(error is TypeError for error, _ in invalid_crop_arguments)
print(f"Oversized target: (11, 8) after padding an (8, 8) image to (10, 10)")
print(f"Constructor cases: {value_error_count} ValueError, {type_error_count} TypeError")
print("PASS: oversized targets and invalid constructor arguments are rejected")

Oversized target: (11, 8) after padding an (8, 8) image to (10, 10)
Constructor cases: 7 ValueError, 12 TypeError
PASS: oversized targets and invalid constructor arguments are rejected


## RandomCrop results

This checklist is generated only after every crop assertion above has passed. Each label is recorded once, avoiding a duplicate expected-results list.

In [34]:
assert len(crop_checks) == 12
assert all(crop_checks.values())

for check in crop_checks:
    print(f"[x] {check}")
print(f"\nAll {len(crop_checks)} requested RandomCrop checks passed.")

[x] padding=0 with target equal to image size returns identical data
[x] 2D (H, W) produces (target_h, target_w)
[x] CHW produces (C, target_h, target_w)
[x] HWC produces (target_h, target_w, C)
[x] NumPy input returns NumPy
[x] Tensor_CP input returns Tensor_CP
[x] Original input remains unchanged
[x] Integer size becomes a square crop
[x] Tuple size produces a rectangular crop
[x] Resetting the same random seed produces the same crop
[x] Target larger than the padded image raises ValueError
[x] Invalid sizes and padding values raise the intended errors

All 12 requested RandomCrop checks passed.


## Compose

`Compose` must apply transforms in order, preserve the image container type through the built-in transforms, and reject a non-sequence or non-callable entries.

In [35]:
compose_checks = {}
compose_source = np.arange(1, 17).reshape(4, 4)
composed_transform = Compose([
    RandomCrop(size=(4, 4), padding=0),
    RandomHorizontalFlip(p=1),
])
compose_output = composed_transform(compose_source)

assert isinstance(compose_output, np.ndarray)
assert np.array_equal(compose_output, np.flip(compose_source, axis=-1))
assert np.array_equal(compose_source, np.arange(1, 17).reshape(4, 4))

tensor_compose_output = composed_transform(Tensor_CP(compose_source))
assert isinstance(tensor_compose_output, Tensor_CP)
assert np.array_equal(tensor_compose_output.data, np.flip(compose_source, axis=-1))

assert_raises(TypeError, lambda: Compose(RandomHorizontalFlip()))
assert_raises(TypeError, lambda: Compose([RandomHorizontalFlip(), 42]))

compose_checks["Transforms run in the declared order"] = True
compose_checks["NumPy and Tensor_CP container types are preserved"] = True
compose_checks["Inputs remain unchanged"] = True
compose_checks["Invalid transform collections are rejected"] = True

for check in compose_checks:
    print(f"[x] {check}")
print(f"\nAll {len(compose_checks)} Compose checks passed.")

[x] Transforms run in the declared order


[x] NumPy and Tensor_CP container types are preserved
[x] Inputs remain unchanged
[x] Invalid transform collections are rejected

All 4 Compose checks passed.


## PyTorch parity

Both implementations iterate over the same 11 samples without shuffling. The comparison checks batch count, partial-final-batch behavior, shapes, and every value.

In [36]:
torch_dataset = TorchTensorDataset(
    torch.from_numpy(feature_values),
    torch.from_numpy(label_values),
)
torch_loader = TorchDataLoader(torch_dataset, batch_size=4, shuffle=False)

tiny_batches = list(DataLoader(batch_dataset, batch_size=4, shuffle=False))
torch_batches = list(torch_loader)

assert len(tiny_batches) == len(torch_batches) == 3
max_feature_difference = 0.0
max_label_difference = 0.0

for (tiny_features, tiny_labels), (torch_features, torch_labels) in zip(
    tiny_batches, torch_batches
):
    assert tiny_features.shape == tuple(torch_features.shape)
    assert tiny_labels.shape == tuple(torch_labels.shape)
    max_feature_difference = max(
        max_feature_difference,
        float(np.max(np.abs(tiny_features.data - torch_features.numpy()))),
    )
    max_label_difference = max(
        max_label_difference,
        float(np.max(np.abs(tiny_labels.data - torch_labels.numpy()))),
    )

assert max_feature_difference == 0.0
assert max_label_difference == 0.0

print(f"Batch sizes:            {[batch[0].shape[0] for batch in tiny_batches]}")
print(f"Maximum feature error: {max_feature_difference:.1f}")
print(f"Maximum label error:   {max_label_difference:.1f}")
print("PASS: TinyTorch and PyTorch produce identical sequential batches")

Batch sizes:            [4, 4, 3]


Maximum feature error: 0.0
Maximum label error:   0.0
PASS: TinyTorch and PyTorch produce identical sequential batches


## Efficiency

The benchmark iterates through a complete 4,096-sample in-memory dataset with 32 features and `batch_size=64`. Inputs and loader objects are created outside the timed region. Each result is the median milliseconds per epoch after warm-up runs.

In [37]:
def benchmark_epoch(iterate_epoch, *, warmup=3, repeats=9):
    for _ in range(warmup):
        iterate_epoch()

    samples_ms = []
    for _ in range(repeats):
        start = perf_counter()
        iterate_epoch()
        samples_ms.append((perf_counter() - start) * 1_000)
    return median(samples_ms), samples_ms


benchmark_rng = np.random.default_rng(42)
benchmark_features = benchmark_rng.standard_normal((4096, 32)).astype(np.float32)
benchmark_labels = benchmark_rng.integers(0, 10, size=4096).astype(np.float32)

tiny_benchmark_loader = DataLoader(
    TensorDataset(Tensor_CP(benchmark_features), Tensor_CP(benchmark_labels)),
    batch_size=64,
    shuffle=False,
)
torch_benchmark_loader = TorchDataLoader(
    TorchTensorDataset(
        torch.from_numpy(benchmark_features), torch.from_numpy(benchmark_labels)
    ),
    batch_size=64,
    shuffle=False,
    num_workers=0,
)


def consume_tiny_epoch():
    checksum = 0.0
    for batch_features, batch_labels in tiny_benchmark_loader:
        checksum += float(batch_features.data[0, 0]) + float(batch_labels.data[0])
    return checksum


def consume_torch_epoch():
    checksum = 0.0
    for batch_features, batch_labels in torch_benchmark_loader:
        checksum += float(batch_features[0, 0]) + float(batch_labels[0])
    return checksum


tiny_checksum = consume_tiny_epoch()
torch_checksum = consume_torch_epoch()
assert np.isclose(tiny_checksum, torch_checksum)
assert len(tiny_benchmark_loader) == len(torch_benchmark_loader) == 64

tiny_epoch_ms, tiny_samples_ms = benchmark_epoch(consume_tiny_epoch)
torch_epoch_ms, torch_samples_ms = benchmark_epoch(consume_torch_epoch)
epoch_ratio = tiny_epoch_ms / torch_epoch_ms
throughput = len(benchmark_features) / (tiny_epoch_ms / 1_000)

print(f"Dataset:              {len(benchmark_features):,} samples x 32 features")
print(f"Batch size / batches: 64 / {len(tiny_benchmark_loader)}")
print(f"TinyTorch median:     {tiny_epoch_ms:.3f} ms/epoch")
print(f"PyTorch median:       {torch_epoch_ms:.3f} ms/epoch")
print(f"TinyTorch / PyTorch:  {epoch_ratio:.2f}x")
print(f"TinyTorch throughput: {throughput:,.0f} samples/s")
print(f"Checksum difference:  {abs(tiny_checksum - torch_checksum):.1e}")

Dataset:              4,096 samples x 32 features
Batch size / batches: 64 / 64
TinyTorch median:     10.288 ms/epoch
PyTorch median:       20.606 ms/epoch
TinyTorch / PyTorch:  0.50x
TinyTorch throughput: 398,122 samples/s
Checksum difference:  0.0e+00


## Takeaways

- All assertion-based checks cover dataset contracts, batch boundaries, ordering, shuffling, transforms, composition, edge cases, and input validation.
- TinyTorch and PyTorch produce exactly equal values for the sequential parity fixture, including the final partial batch.
- The efficiency result measures this intentionally small, single-process educational implementation. PyTorch's loader has broader functionality and different optimization tradeoffs, so the ratio should be interpreted only for the stated CPU workload.
- Re-run the notebook on the target machine before quoting timing numbers elsewhere, because CPU load and library versions affect microbenchmarks.